# Hyper-optimisation — Adult Income (UCI)

Ce notebook règle les hyperparamètres des modèles avec **Optuna** (`HyperparameterTuner`),
sélectionne le **modèle final** par validation croisée, trace sa **courbe d'apprentissage**
et choisit le **seuil de décision** sur des prédictions **hors-pli**.

> ⚠️ **Tout se fait sur le train (`adult.data`).** Le jeu de test (`adult.test`) n'est
> **jamais** touché ici : l'évaluation finale vit dans `evaluation.ipynb`. Cette séparation
> garantit qu'aucune décision (modèle, seuil) n'est prise en regardant le test.

Chaîne : `DataLoader` → `PreprocessingPipeline` → `ModelRegistry` → `HyperparameterTuner` (Optuna)
→ sélection CV → seuil hors-pli → sauvegarde du modèle final pour `evaluation.ipynb`.


In [ ]:
# Recharge automatique des modules src/ modifiés sans restart du kernel
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# Permet d'importer le package src/ depuis le dossier notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from adult_income import (
    DataLoader,
    HyperparameterTuner,
    ModelEvaluator,
    ModelRegistry,
    PreprocessingPipeline,
)

STORAGE = ROOT / "artifacts" / "optuna"
print("Imports OK")

## 1. Données & prétraitement

In [ ]:
# Train UNIQUEMENT (load_train) : le test reste de côté jusqu'à evaluation.ipynb.
loader = DataLoader()
X_train, y_train = loader.load_train()

pp = PreprocessingPipeline(capital_treatment="naive")
y_train_enc = pp.fit_transform_target(y_train)  # <=50K -> 0, >50K -> 1

print(f"Train : {X_train.shape[0]:,} obs, {X_train.shape[1]} variables")
print(f"Prévalence >50K : {y_train_enc.mean():.1%}")

## 2. Hyper-optimisation Optuna

Chaque modèle disposant d'un espace de recherche est optimisé par une étude **Optuna**
(TPE + pruning médian) qui **maximise la PR-AUC** (`average_precision`) en CV 5 plis sur le
train. La PR-AUC est privilégiée au ROC-AUC car elle reflète mieux la performance sur la
classe minoritaire (`>50K`, ~24 %).

Les résultats (50 trials/modèle) sont **sauvegardés** dans `artifacts/optuna/best_params.json`.
Par défaut on **recharge** ces paramètres pour éviter de relancer une optimisation longue
(`RUN_TUNING = False`). Mettre `RUN_TUNING = True` rejoue les études (et régénère la base
SQLite nécessaire aux visualisations Optuna).


In [ ]:
RUN_TUNING = False  # True -> relance Optuna (long) + régénère la base SQLite

TUNABLE = HyperparameterTuner.tunable_models()
print(f"Modèles optimisables ({len(TUNABLE)}) : {TUNABLE}")

if RUN_TUNING:
    tuner = HyperparameterTuner(
        n_trials=50, cv=5, scoring="average_precision", n_jobs=4, storage=STORAGE
    )
    for name in TUNABLE:
        print(f"Tuning {name}...")
        tuner.tune(name, pp.build_pipeline(ModelRegistry.get(name)), X_train, y_train_enc)

# Rechargement léger depuis le JSON (pas besoin d'Optuna)
best = HyperparameterTuner.load_best_params(STORAGE)
HyperparameterTuner.summary_table(best).round(4)

In [ ]:
# Reconstruire les pipelines avec les meilleurs hyperparamètres trouvés.
tuned_pipelines = {
    n: HyperparameterTuner.build_tuned_pipeline(pp, n, info["best_params"])
    for n, info in best.items()
}
print("Pipelines optimisés :", list(tuned_pipelines))

## 3. Visualisations Optuna (optionnel)

Historique d'optimisation et importance des hyperparamètres. Ces graphes nécessitent la
**base SQLite** des études (générée seulement si `RUN_TUNING = True`). Sinon, la cellule
l'indique et passe.


In [ ]:
try:
    viz = HyperparameterTuner(storage=STORAGE)
    saved = viz.list_saved_studies()
    if any(s.startswith("xgboost") for s in saved):
        viz.load_study("xgboost")
        display(viz.plot_optimization_history("xgboost"))
        display(viz.plot_param_importances("xgboost"))
    else:
        print("Études Optuna absentes du disque — relancer avec RUN_TUNING=True.")
except Exception as e:
    print("Visualisations indisponibles :", e)

## 4. Sélection multicritère du modèle final

On compare les modèles optimisés en **validation croisée sur le train** (toujours pas de test).
La **PR-AUC** est prioritaire ; les autres métriques départagent les modèles proches. On
retient le meilleur représentant pour la suite.


In [ ]:
selector = ModelEvaluator(cv=5, random_state=42)
selector.cross_validate_all(tuned_pipelines, X_train, y_train_enc)

selection_table = selector.cv_comparison_table(sort_by="average_precision")
selection_table[[
    "model", "average_precision_mean", "f1_mean", "roc_auc_mean",
    "precision_mean", "recall_mean", "accuracy_mean",
]].round(4)

In [ ]:
best_model_name = selection_table.iloc[0]["model"]
final_params = best[best_model_name]["best_params"]
final_pipeline = HyperparameterTuner.build_tuned_pipeline(pp, best_model_name, final_params)
final_ap = float(selection_table.iloc[0]["average_precision_mean"])

print(f"Modèle final retenu : {best_model_name}")
print(f"PR-AUC (CV train)   : {final_ap:.4f}")
print(f"Hyperparamètres     : {final_params}")

## 5. Courbe d'apprentissage

Évolution de la PR-AUC (entraînement vs validation croisée) selon la taille du jeu
d'entraînement. Un écart qui se resserre indique une bonne généralisation ; un plateau de la
courbe de validation suggère que plus de données n'aiderait plus.


In [ ]:
selector.plot_learning_curve(
    final_pipeline, X_train, y_train_enc,
    scoring="average_precision", model_name=best_model_name,
)
plt.show()

## 6. Choix du seuil de décision (hors-pli)

Le seuil par défaut (0,5) n'est pas forcément optimal en déséquilibre. On le règle sur des
prédictions **hors-pli** (`cross_val_predict`) du train — donc **sans fuite** : aucune
observation n'est notée par un modèle qui l'a vue. On maximise le **F1** (compromis
précision/rappel). Ce seuil sera appliqué tel quel sur le test dans `evaluation.ipynb`.


In [ ]:
y_proba_oof = selector.out_of_fold_probabilities(final_pipeline, X_train, y_train_enc)
best_threshold, threshold_tbl = selector.find_best_threshold(y_train_enc, y_proba_oof, metric="f1")

print(f"Seuil optimal (F1, hors-pli) : {best_threshold:.3f}")
threshold_tbl.sort_values("f1", ascending=False).head(1).round(4)

In [ ]:
selector.plot_threshold_curves(threshold_tbl, best_threshold)
plt.show()

## 7. Sauvegarde pour l'évaluation finale

On sérialise le **modèle retenu**, ses **hyperparamètres** et le **seuil** dans
`artifacts/final_model.json`. Le notebook `evaluation.ipynb` les recharge, réentraîne sur tout
le train, et n'évalue **qu'alors** sur `adult.test`.


In [ ]:
final_spec = {
    "model_name": best_model_name,
    "best_params": final_params,
    "threshold": best_threshold,
    "scoring": "average_precision",
    "cv_average_precision": final_ap,
}
out_path = ROOT / "artifacts" / "final_model.json"
out_path.write_text(json.dumps(final_spec, indent=2))
print("Sauvegardé ->", out_path)
final_spec